In [4]:
# ==============================================================================
# 🌟 TERMINAL FINANCIERA INSTITUCIONAL & TRADING
# ==============================================================================

!pip install -q feedparser yfinance google-genai pandas numpy ipywidgets

import time
import feedparser
import yfinance as yf
import pandas as pd
import numpy as np
from google import genai
from google.genai import types
import urllib.parse
from IPython.display import Markdown, display, clear_output, HTML
import ipywidgets as widgets

# Estilos CSS Modernos FinTech
css_estilos = HTML("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700&display=swap');
.fintech-container {
    font-family: 'Inter', sans-serif;
    background: linear-gradient(135deg, #0f172a 0%, #1e293b 100%);
    border-radius: 16px; padding: 24px;
    box-shadow: 0 10px 25px -5px rgba(0, 0, 0, 0.4);
    color: #f8fafc; border: 1px solid rgba(255, 255, 255, 0.1);
}
.fintech-header {
    font-size: 22px; font-weight: 700; color: #38bdf8;
    margin-bottom: 16px; display: flex; align-items: center; gap: 10px;
    border-bottom: 2px solid rgba(56, 189, 248, 0.2); padding-bottom: 10px;
}
input, select, textarea {
    background-color: #1e293b !important; color: #f8fafc !important;
    border: 1px solid #334155 !important; border-radius: 8px !important;
    padding: 8px 12px !important; font-family: 'Inter', sans-serif !important;
}
.custom-btn {
    background: linear-gradient(135deg, #0284c7 0%, #0369a1 100%) !important;
    color: white !important; font-weight: 600 !important; border-radius: 8px !important;
    border: none !important; padding: 10px 20px !important; cursor: pointer !important;
}
.custom-btn-success {
    background: linear-gradient(135deg, #059669 0%, #047857 100%) !important;
}
</style>
""")
display(css_estilos)

# Componentes de Interfaz UI
input_api_key = widgets.Password(description="🔑 API Key:", placeholder="Gemini API Key", layout=widgets.Layout(width='450px'))
input_tema_macro = widgets.Text(description="🔍 Tema/País:", placeholder="Ej. Japón, Euro, Oro, FED", layout=widgets.Layout(width='400px'))
input_ticker_macro = widgets.Text(description="📊 Ticker (Opc):", placeholder="Ej. EURUSD=X, GC=F", layout=widgets.Layout(width='400px'))

btn_analizar_macro = widgets.Button(description="🏛️ Generar Informe Macroeconómico", layout=widgets.Layout(width='320px', height='42px'))
btn_analizar_macro.class_name = 'custom-btn'

input_ticker_trading = widgets.Text(description="📊 Activo:", placeholder="Ej. EURUSD, BTC-USD, NVDA", layout=widgets.Layout(width='400px'))
select_temporalidad = widgets.Dropdown(
    options=['5m', '15m', '30m', '1h', '1d'],
    value='15m',
    description='⏱️ Temporal:',
    layout=widgets.Layout(width='300px')
)

btn_analizar_trading = widgets.Button(description="⚡ Analizar Estructura de Trading", layout=widgets.Layout(width='320px', height='42px'))
btn_analizar_trading.class_name = 'custom-btn custom-btn-success'

output_area = widgets.Output()

tab_macro = widgets.VBox([
    widgets.HTML("<p style='color: #94a3b8; font-size: 13px;'>Análisis integral de economías, divisas, bancos centrales y geopolítica.</p>"),
    input_tema_macro, input_ticker_macro, widgets.HTML("<br>"), btn_analizar_macro
], layout=widgets.Layout(padding='10px'))

tab_trading = widgets.VBox([
    widgets.HTML("<p style='color: #94a3b8; font-size: 13px;'>Análisis cuantitativo de microestructura, S/R, RSI, Volumen y Medias Móviles (incluye 5m).</p>"),
    input_ticker_trading, select_temporalidad, widgets.HTML("<br>"), btn_analizar_trading
], layout=widgets.Layout(padding='10px'))

tabs = widgets.Tab(children=[tab_macro, tab_trading])
tabs.set_title(0, '🏛️ Macro & Divisas')
tabs.set_title(1, '⚡ Trading Táctico')

dashboard_html = widgets.HTML("""
<div class="fintech-container">
    <div class="fintech-header"><span>🌐</span> <span>Terminal Cuantitativa & Macroeconómica AI (Pro)</span></div>
</div>
""")

ui_maestro = widgets.VBox([dashboard_html, widgets.HTML("<br>"), input_api_key, widgets.HTML("<hr style='border-color: #334155;'>"), tabs, widgets.HTML("<hr style='border-color: #334155;'>"), output_area])

# Lógica de Ejecución Macro
def ejecutar_macro(b):
    with output_area:
        clear_output()
        api_k = input_api_key.value.strip()
        tema = input_tema_macro.value.strip()

        if not api_k or not tema:
            print("❌ Error: Ingresa tu API Key y el Tema a analizar.")
            return

        print("⚡ Recopilando inteligencia de noticias globales (Google News)...")
        noticias_str = "Sin noticias recientes."
        try:
            busqueda_url = urllib.parse.quote(f"{tema} economía finanzas mercado")
            url = f"https://news.google.com/rss/search?q={busqueda_url}&hl=es-419&gl=US"
            feed = feedparser.parse(url)
            titulares = [f"- {e.title}" for e in feed.entries[:8]]
            if titulares: noticias_str = "\n".join(titulares)
        except Exception as e:
            print(f"⚠️ Aviso en noticias: {e}")

        datos_str = "Sin ticker provisto."
        ticker_macro_val = input_ticker_macro.value.strip().upper()
        if ticker_macro_val:
            if ticker_macro_val in ['EURUSD', 'GBPUSD', 'USDJPY', 'AUDUSD', 'USDCAD'] and not ticker_macro_val.endswith('=X'):
                ticker_macro_val += '=X'
            print(f"📊 Extrayendo métricas de mercado para '{ticker_macro_val}'...")
            try:
                hist = yf.Ticker(ticker_macro_val).history(period="1mo")
                if not hist.empty:
                    close_s = hist['Close'].iloc[:, 0] if isinstance(hist['Close'], pd.DataFrame) else hist['Close']
                    p_act = round(float(close_s.iloc[-1]), 2)
                    p_max = round(float(hist['High'].max() if not isinstance(hist['High'], pd.DataFrame) else hist['High'].iloc[:, 0].max()), 2)
                    p_min = round(float(hist['Low'].min() if not isinstance(hist['Low'], pd.DataFrame) else hist['Low'].iloc[:, 0].min()), 2)
                    tend = "ALCISTA 🟢" if p_act > float(close_s.iloc[0]) else "BAJISTA 🔴"
                    datos_str = f"Activo: {ticker_macro_val} | Cierre Actual: {p_act} | Máximo Mensual: {p_max} | Mínimo Mensual: {p_min} | Tendencia Macro: {tend}"
            except Exception as e:
                print(f"⚠️ Aviso en Yahoo Finance: {e}")

        prompt = f"""
        Actúa como un Estratega Macroeconómico Senior de un fondo de inversión global.
        Analiza a máxima profundidad y rigor los siguientes datos:
        - Tema / Objetivo: {tema}
        - Datos Cuantitativos de Mercado: {datos_str}
        - Inteligencia Cualitativa (Noticias):
        {noticias_str}

        Estructura el informe estrictamente con Markdown y los siguientes encabezados:
        ### 📌 1. Resumen Ejecutivo (TL;DR)
        ### 📊 2. Entorno Macroeconómico, Monetario y Cambiario
        ### 🌐 3. Factores Geopolíticos y Dinámica de Noticias
        ### 🔮 4. Proyección de Escenarios (Corto y Mediano Plazo)
        """
        llamar_gemini(api_k, prompt, f"Macro_{tema.replace(' ', '_')}.txt")

# Lógica de Ejecución Trading (Con 5m y Dirección de Mercado)
def ejecutar_trading(b):
    with output_area:
        clear_output()
        api_k = input_api_key.value.strip()
        ticker = input_ticker_trading.value.strip().upper()
        temp = select_temporalidad.value

        if not api_k or not ticker:
            print("❌ Error: Ingresa tu API Key y el Símbolo del Activo.")
            return

        if ticker in ['EURUSD', 'GBPUSD', 'USDJPY', 'AUDUSD', 'USDCAD', 'NZDUSD', 'EURJPY', 'GBPJPY'] and not ticker.endswith('=X'):
            ticker += '=X'

        print(f"⚡ Procesando microestructura avanzada en temporalidad [{temp}] para '{ticker}'...")

        # Mapeo de periodos optimizado para velas rápidas (incluyendo 5m)
        periodo_map = {
            "5m": "1d",
            "15m": "5d",
            "30m": "5d",
            "1h": "1mo",
            "1d": "3mo"
        }
        periodo = periodo_map.get(temp, "5d")

        try:
            hist = yf.Ticker(ticker).history(period=periodo, interval=temp)
            if hist.empty:
                print(f"❌ No se encontraron datos para '{ticker}' en la temporalidad {temp}.")
                return

            close_col = hist['Close'].iloc[:, 0] if isinstance(hist['Close'], pd.DataFrame) else hist['Close']
            high_col = hist['High'].iloc[:, 0] if isinstance(hist['High'], pd.DataFrame) else hist['High']
            low_col = hist['Low'].iloc[:, 0] if isinstance(hist['Low'], pd.DataFrame) else hist['Low']
            vol_col = hist['Volume'].iloc[:, 0] if isinstance(hist['Volume'], pd.DataFrame) else hist['Volume']

            p_act = round(float(close_col.iloc[-1]), 2)
            resis = round(float(high_col.tail(15).max()), 2)
            sopor = round(float(low_col.tail(15).min()), 2)
            vol_promedio = int(vol_col.mean()) if not vol_col.empty else 0
            vol_actual = int(vol_col.iloc[-1]) if not vol_col.empty else 0

            # Medias móviles y RSI
            sma_20 = round(float(close_col.rolling(window=20).mean().iloc[-1]), 2) if len(close_col) >= 20 else p_act

            delta = close_col.diff()
            gain = (delta.where(delta > 0, 0)).rolling(14).mean()
            loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
            rs = gain / loss
            rsi_raw = (100 - (100 / (1 + rs))).iloc[-1]
            rsi = round(float(rsi_raw.item() if hasattr(rsi_raw, 'item') else rsi_raw), 2) if not pd.isna(rsi_raw) else 50.0

            # Determinación cuantitativa preliminar de tendencia para ayudar al modelo
            tendencia_precio = "ALCISTA 🟢" if p_act > sma_20 else "BAJISTA 🔴"

            datos_tec = (
                f"• Activo: {ticker}\n"
                f"• Temporalidad de Operación: {temp}\n"
                f"• Precio de Cierre Actual: {p_act}\n"
                f"• Media Móvil Simple (SMA 20): {sma_20}\n"
                f"• Resistencia Clave Reciente: {resis}\n"
                f"• Soporte Clave Reciente: {sopor}\n"
                f"• Indicador RSI (14): {rsi}\n"
                f"• Volumen Actual vs Promedio: {vol_actual:,} / {vol_promedio:,}\n"
                f"• Sesgo Técnico Preliminar: {tendencia_precio}"
            )
        except Exception as e:
            print(f"❌ Error técnico al calcular indicadores: {str(e)}")
            return

        prompt = f"""
        Actúa como un Trader Cuantitativo Senior y Especialista en Acción del Precio en temporalidad de {temp}.
        Analiza a máxima profundidad la siguiente microestructura técnica:
        {datos_tec}

        INSTRUCCIONES CRITICAS:
        1. Debes incluir obligatoriamente al inicio del reporte un titular claro que determine la dirección del mercado basada en los datos técnicos suministrados.
        2. Mantén los encabezados exactos solicitados a continuación.

        Estructura la salida estrictamente con Markdown y los siguientes encabezados:
        ### 🧭 DIRECCIÓN DEL MERCADO: [INDICAR CLARAMENTE SI ES ALCISTA 🟢, BAJISTA 🔴 O LATERAL ⚖️]
        ### ⚡ 1. Contexto de Precio, Momentum y Medias Móviles ({temp})
        ### 🛡️ 2. Niveles Operativos Críticos (Soportes y Resistencias)
        ### 📊 3. Lectura Profunda de Indicadores (RSI y Volumen)
        ### 🎯 4. Planteamiento Táctico (Zona de Entrada, Stop Loss y Take Profit) y Gestión de Riesgo
        """
        llamar_gemini(api_k, prompt, f"Trading_{ticker.replace('=', '_')}_{temp}.txt")

def llamar_gemini(api_key, prompt, nombre_archivo):
    client = genai.Client(api_key=api_key)
    modelo = 'gemini-3.8-flash'
    respuesta = None
    max_intentos = 6

    for intento in range(1, max_intentos + 1):
        try:
            print(f"🤖 Conectando con Gemini AI (Intento {intento} de {max_intentos})...")
            respuesta = client.models.generate_content(
                model=modelo,
                contents=prompt,
                config=types.GenerateContentConfig(temperature=0.2)
            )
            if respuesta and hasattr(respuesta, 'text') and respuesta.text:
                break
        except Exception as e:
            error_str = str(e)
            print(f"⚠️ Aviso en intento {intento}: {error_str}")
            if "503" in error_str or "unavailable" in error_str.lower() or "rate" in error_str.lower():
                if intento < max_intentos:
                    tiempo_espera = intento * 2
                    print(f"⏳ Reintentando en {tiempo_espera} segundos...")
                    time.sleep(tiempo_espera)
            else:
                break

    clear_output()
    if respuesta and hasattr(respuesta, 'text') and respuesta.text:
        texto = respuesta.text
        with open(nombre_archivo, "w", encoding="utf-8") as f:
            f.write(texto)

        display(HTML(f"""
        <div class='fintech-container' style='border: 1px solid #059669;'>
            <div style='color: #34d399; font-weight: 600;'>✨ ¡Análisis institucional completo generado y guardado en '{nombre_archivo}'!</div>
        </div>
        <br>
        """))
        display(Markdown(texto))
    else:
        print("❌ No se pudo completar la conexión por alta congestión temporal en los servidores de IA. Vuelve a hacer clic en el botón.")

btn_analizar_macro.on_click(ejecutar_macro)
btn_analizar_trading.on_click(ejecutar_trading)

display(ui_maestro)